# 🔬 DiabetAI v8 - SHAP Diagnostic Pipeline

**Problem:** Model predicting conservative doses (0.77U vs expected 1.67U)

**Goal:** Identify which features are "pushing down" the predictions

In [ ]:
!pip install shap polars xgboost matplotlib -U -q

import shap
import polars as pl
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import json

from google.colab import drive
drive.mount('/content/drive')

print(f"✅ SHAP: {shap.__version__}")
print(f"✅ XGBoost: {xgb.__version__}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS"
MODEL_PATH = f"{BASE_PATH}/models/xgboost_clinical_v8.json"
TEST_FILE = f"{BASE_PATH}/data/processed/test_ready_real_world_v7.parquet"

# v8 Feature List (17 columns - EXACT ORDER)
FEATURES = [
    'CGM',              # 0
    'carbs',            # 1
    'iob',              # 2
    'est_tdd',          # 3
    'est_isf',          # 4
    'est_icr',          # 5
    'cgm_delta_5min',   # 6
    'cgm_delta_15min',  # 7
    'cgm_velocity',     # 8
    'basal',            # 9
    'basal_delta_5min', # 10
    'time_sin',         # 11
    'time_cos',         # 12
    'is_weekend',       # 13
    'age',              # 14
    'weight',           # 15
    'gender_encoded'    # 16
]

print(f"📋 Features: {len(FEATURES)}")

In [ ]:
# Load Model
print("📂 Loading v8 model...")
model = xgb.Booster()
model.load_model(MODEL_PATH)
print(f"✅ Model loaded: {MODEL_PATH}")

# Load Test Data
print("\n📂 Loading test data...")
df_test = pl.read_parquet(TEST_FILE)
print(f"✅ Test: {len(df_test):,} rows")

# Prepare features
X_test = df_test.select(FEATURES).to_numpy().astype(np.float32)
y_test = df_test.select('bolus').to_numpy().ravel().astype(np.float32)

print(f"\n📊 X_test: {X_test.shape}")

## Step 1: Initialize SHAP TreeExplainer

In [ ]:
print("="*70)
print("🔬 INITIALIZING SHAP EXPLAINER")
print("="*70)

# TreeExplainer is optimized for tree-based models
# Works well with tree_method='hist'
explainer = shap.TreeExplainer(model)

print(f"\n✅ TreeExplainer initialized")
print(f"   Expected value (base): {explainer.expected_value:.4f}")

In [ ]:
# Use last 1000 samples for SHAP analysis (speed optimization)
print("\n🔄 Computing SHAP values for last 1000 samples...")

sample_size = 1000
X_sample = X_test[-sample_size:]
y_sample = y_test[-sample_size:]

# Convert to DMatrix for prediction
dmatrix = xgb.DMatrix(X_sample, feature_names=FEATURES)
predictions = model.predict(dmatrix)

# Get SHAP values
shap_values = explainer.shap_values(X_sample)

print(f"\n✅ SHAP values computed: {shap_values.shape}")

## Step 2: Global Feature Importance (Summary Plot)

In [ ]:
print("="*70)
print("📊 GLOBAL SHAP SUMMARY")
print("="*70)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, show=False)
plt.title("v8 Model - SHAP Feature Impact")
plt.tight_layout()
plt.savefig(f"{BASE_PATH}/models/v8_shap_summary.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Summary plot saved!")

In [ ]:
# Bar plot for mean absolute SHAP values
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, plot_type='bar', show=False)
plt.title("v8 Model - Mean |SHAP| (Feature Importance)")
plt.tight_layout()
plt.savefig(f"{BASE_PATH}/models/v8_shap_importance.png", dpi=150, bbox_inches='tight')
plt.show()

## Step 3: Single Prediction Explainer Function

In [ ]:
def explain_prediction(input_vector, math_dose=None, plot=True):
    """
    Explain a single prediction using SHAP.
    
    Args:
        input_vector: np.array of shape (1, 17) or (17,)
        math_dose: Optional clinical math calculation for comparison
        plot: Whether to show waterfall plot
        
    Returns:
        dict with prediction, shap_values, and deviation analysis
    """
    # Ensure correct shape
    if input_vector.ndim == 1:
        input_vector = input_vector.reshape(1, -1)
    
    input_vector = input_vector.astype(np.float32)
    
    # Get prediction
    dmatrix = xgb.DMatrix(input_vector, feature_names=FEATURES)
    prediction = float(model.predict(dmatrix)[0])
    
    # Get SHAP values
    shap_vals = explainer.shap_values(input_vector)[0]
    base_value = explainer.expected_value
    
    # Create result dict
    result = {
        'prediction': prediction,
        'base_value': base_value,
        'shap_values': dict(zip(FEATURES, shap_vals)),
        'input_values': dict(zip(FEATURES, input_vector[0])),
        'top_pushers': [],
        'top_suppressors': [],
        'deviation_warning': None
    }
    
    # Sort features by SHAP contribution
    sorted_features = sorted(zip(FEATURES, shap_vals), key=lambda x: x[1])
    
    # Top 3 features pushing DOWN (negative SHAP)
    result['top_suppressors'] = [
        {'feature': f, 'shap': v, 'value': input_vector[0][FEATURES.index(f)]}
        for f, v in sorted_features[:3] if v < 0
    ]
    
    # Top 3 features pushing UP (positive SHAP)
    result['top_pushers'] = [
        {'feature': f, 'shap': v, 'value': input_vector[0][FEATURES.index(f)]}
        for f, v in sorted_features[-3:][::-1] if v > 0
    ]
    
    # Residual analysis
    if math_dose is not None:
        residual = abs(math_dose - prediction)
        result['math_dose'] = math_dose
        result['residual'] = residual
        
        if residual > 0.5:
            result['deviation_warning'] = {
                'level': 'HIGH' if residual > 1.0 else 'MODERATE',
                'residual': residual,
                'direction': 'UNDER' if prediction < math_dose else 'OVER',
                'top_causes': result['top_suppressors'] if prediction < math_dose else result['top_pushers']
            }
    
    # Plot
    if plot:
        print("\n" + "="*60)
        print("🔬 SHAP EXPLANATION")
        print("="*60)
        print(f"\n📊 Prediction: {prediction:.3f}U")
        print(f"📊 Base value: {base_value:.3f}U")
        
        if math_dose:
            print(f"📊 Math dose: {math_dose:.3f}U")
            print(f"📊 Residual: {residual:.3f}U")
        
        print("\n🔴 Top Suppressors (pushing DOWN):")
        for item in result['top_suppressors']:
            print(f"   {item['feature']}: {item['shap']:+.4f} (value={item['value']:.2f})")
        
        print("\n🟢 Top Pushers (pushing UP):")
        for item in result['top_pushers']:
            print(f"   {item['feature']}: {item['shap']:+.4f} (value={item['value']:.2f})")
        
        if result['deviation_warning']:
            warn = result['deviation_warning']
            print(f"\n⚠️ {warn['level']} DEVIATION WARNING!")
            print(f"   Direction: {warn['direction']}-PREDICTION")
            print(f"   Residual: {warn['residual']:.3f}U")
        
        # Waterfall plot
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_vals,
                base_values=base_value,
                data=input_vector[0],
                feature_names=FEATURES
            ),
            show=False
        )
        plt.title(f"Prediction: {prediction:.2f}U | Math: {math_dose:.2f}U" if math_dose else f"Prediction: {prediction:.2f}U")
        plt.tight_layout()
        plt.show()
    
    return result

## Step 4: Test the Explainer

In [ ]:
# Create a test case: 120 mg/dL, 50g carbs, 0 IOB
# Expected Math: 50/10 = 5.0U (with ICR=10)

test_input = np.array([
    120.0,   # CGM
    50.0,    # carbs
    0.0,     # iob
    40.0,    # est_tdd
    30.0,    # est_isf
    10.0,    # est_icr
    0.0,     # cgm_delta_5min
    0.0,     # cgm_delta_15min
    0.0,     # cgm_velocity
    1.0,     # basal
    0.0,     # basal_delta_5min
    0.0,     # time_sin
    1.0,     # time_cos
    0.0,     # is_weekend
    30.0,    # age
    70.0,    # weight
    0.0      # gender_encoded
], dtype=np.float32)

# Math calculation: carbs/ICR + (CGM-target)/ISF
math_dose = (50.0 / 10.0) + max(0, (120.0 - 100.0) / 30.0)
print(f"🧮 Math Dose: {50}/10 + (120-100)/30 = {math_dose:.2f}U")

result = explain_prediction(test_input, math_dose=math_dose)

In [ ]:
# Another test: Conservative scenario (what user reported)
# Should predict ~1.67U but model gives 0.77U

conservative_input = np.array([
    100.0,   # CGM (in range)
    20.0,    # carbs (small meal)
    0.0,     # iob (zero!)
    35.0,    # est_tdd
    45.0,    # est_isf
    12.0,    # est_icr
    2.0,     # cgm_delta_5min (slight rise)
    6.0,     # cgm_delta_15min
    0.4,     # cgm_velocity
    0.8,     # basal
    0.0,     # basal_delta_5min
    0.5,     # time_sin (afternoon)
    0.866,   # time_cos
    0.0,     # is_weekend
    35.0,    # age
    75.0,    # weight
    0.0      # gender_encoded
], dtype=np.float32)

# Math: 20/12 = 1.67U
math_dose_conservative = 20.0 / 12.0
print(f"\n🧮 Expected Math: 20/12 = {math_dose_conservative:.2f}U")

result_conservative = explain_prediction(conservative_input, math_dose=math_dose_conservative)

## Step 5: Batch Analysis - Find Under-Prediction Patterns

In [ ]:
print("="*70)
print("📊 BATCH ANALYSIS: Finding Under-Prediction Patterns")
print("="*70)

# Calculate residuals for test set
dmatrix_all = xgb.DMatrix(X_sample, feature_names=FEATURES)
preds = model.predict(dmatrix_all)

# Find cases where prediction is much lower than actual
residuals = y_sample - preds
under_predicted = residuals > 0.5  # Model under-predicted by > 0.5U

print(f"\n📊 Under-predictions (>0.5U gap): {under_predicted.sum()} / {len(residuals)}")
print(f"   Percentage: {100 * under_predicted.mean():.1f}%")

# Average SHAP values for under-predicted cases
if under_predicted.sum() > 0:
    under_shap = shap_values[under_predicted]
    mean_shap_under = under_shap.mean(axis=0)
    
    print("\n🔴 Features causing under-prediction (avg SHAP):")
    sorted_under = sorted(zip(FEATURES, mean_shap_under), key=lambda x: x[1])
    for feat, val in sorted_under[:5]:
        print(f"   {feat}: {val:+.4f}")

In [ ]:
# Save explainer for Streamlit integration
import pickle

explainer_data = {
    'expected_value': float(explainer.expected_value),
    'feature_names': FEATURES,
    'model_path': MODEL_PATH
}

with open(f"{BASE_PATH}/models/shap_explainer_config.json", 'w') as f:
    json.dump(explainer_data, f, indent=2)

print("\n✅ SHAP config saved for Streamlit integration!")

In [ ]:
# Final summary
print("\n" + "="*70)
print("🎉 SHAP DIAGNOSTIC COMPLETE")
print("="*70)
print(f"\n📊 Model Base Value: {explainer.expected_value:.4f}")
print(f"📊 Sample Size: {sample_size}")
print(f"📊 Under-predictions: {100 * under_predicted.mean():.1f}%")
print(f"\n💾 Saved:")
print(f"   - v8_shap_summary.png")
print(f"   - v8_shap_importance.png")
print(f"   - shap_explainer_config.json")